# BZP Data Exploration

Systematic profiling of raw BZP notices to understand the data before
building the gold layer.

**Setup:** Run `batch_fetch.py` first:
```bash
cd E:\git_projects\procurement-watchdog-api-exploration
python E:\git_projects\procurement-watchdog-lakehouse\scripts\batch_fetch.py 2025-10-01 2025-12-31
```

**Findings are recorded in** `docs/data_profile.md`.

In [10]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

sys.path.insert(0, str(Path("../src").resolve()))
from procurement.silver.html_parser import parse_cpv_codes, parse_html

DATA_DIR = Path(r"E:\git_projects\procurement-watchdog-api-exploration\data")
RAW_DIR = DATA_DIR / "raw"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

## 1. Load raw data

In [11]:
raw_files = sorted(RAW_DIR.glob("bzp_*.json"))
print(f"Found {len(raw_files)} raw files")

frames = []
for f in raw_files:
    records = json.loads(f.read_text(encoding="utf-8"))
    df = pd.DataFrame(records)
    df["_file_date"] = f.stem.replace("bzp_", "")
    frames.append(df)

df = pd.concat(frames, ignore_index=True)
df["_pub_dt"] = pd.to_datetime(df["publicationDate"])
df["_weekday"] = df["_pub_dt"].dt.day_name()
print(f"Total records: {len(df):,}  |  Date range: {df['_file_date'].min()} to {df['_file_date'].max()}")

Found 27 raw files
Total records: 97,349  |  Date range: 2025-10-01 to 2025-10-27


## 2. Temporal patterns

In [12]:
daily = df.groupby("_file_date").agg(
    records=("objectId", "count"),
    weekday=("_weekday", "first"),
).reset_index()

print("Daily record counts:")
print(daily[["_file_date", "weekday", "records"]].to_string(index=False))

print(f"\nWeekday mean: {daily.loc[~daily['weekday'].isin(['Saturday','Sunday']), 'records'].mean():.0f}")
print(f"Weekend mean: {daily.loc[daily['weekday'].isin(['Saturday','Sunday']), 'records'].mean():.0f}")

Daily record counts:
_file_date   weekday  records
2025-10-01 Wednesday     5195
2025-10-02  Thursday     4912
2025-10-03    Friday     2389
2025-10-04  Saturday      101
2025-10-05    Sunday     2346
2025-10-06    Monday     4783
2025-10-07   Tuesday     4876
2025-10-08 Wednesday     4884
2025-10-09  Thursday     4939
2025-10-10    Friday     2487
2025-10-11  Saturday      123
2025-10-12    Sunday     2475
2025-10-13    Monday     5032
2025-10-14   Tuesday     5245
2025-10-15 Wednesday     5321
2025-10-16  Thursday     5279
2025-10-17    Friday     2613
2025-10-18  Saturday       98
2025-10-19    Sunday     2773
2025-10-20    Monday     5385
2025-10-21   Tuesday     5450
2025-10-22 Wednesday     5415
2025-10-23  Thursday     4999
2025-10-24    Friday     2413
2025-10-25  Saturday      117
2025-10-26    Sunday     2493
2025-10-27    Monday     5206

Weekday mean: 4570
Weekend mean: 1316


In [13]:
# Publication hour distribution (are there patterns?)
df["_pub_hour"] = df["_pub_dt"].dt.hour
print("Publications by hour of day:")
print(df["_pub_hour"].value_counts().sort_index())

Publications by hour of day:
_pub_hour
0         9
1         2
2         6
3        24
4       212
5      2400
6      6931
7      9894
8     10689
9     12469
10    13250
11    14441
12    13635
13     6797
14     2203
15      967
16      727
17      735
18      584
19      544
20      407
21      314
22       85
23       24
Name: count, dtype: int64


## 3. Notice type taxonomy

Which fields are populated for each notice type? This drives what the
silver and gold layers can extract.

In [14]:
type_counts = df["noticeType"].value_counts()
print("Notice type volumes:")
for t, c in type_counts.items():
    print(f"  {t:40s} {c:>6,}  ({c/len(df)*100:4.1f}%)")

Notice type volumes:
  ContractPerformingNotice                 32,940  (33.8%)
  ContractNotice                           25,917  (26.6%)
  TenderResultNotice                       23,675  (24.3%)
  NoticeUpdateNotice                       12,132  (12.5%)
  AgreementUpdateNotice                     1,247  ( 1.3%)
  AgreementIntentionNotice                    850  ( 0.9%)
  SmallContractNotice                         549  ( 0.6%)
  CircumstancesFulfillmentNotice               34  ( 0.0%)
  CompetitionNotice                             4  ( 0.0%)
  ConcessionNotice                              1  ( 0.0%)


In [15]:
# Field presence (non-null rate) by notice type — the key interdependency table
conditional_fields = [
    "orderType", "tenderType", "tenderId", "submittingOffersDate",
    "procedureResult", "contractors", "isTenderAmountBelowEU",
]

presence = (
    df.groupby("noticeType")[conditional_fields]
    .apply(lambda g: g.notna().mean() * 100)
)
print("Field presence (%) by notice type:")
print(presence.round(0).astype(int).to_string())

Field presence (%) by notice type:
                                orderType  tenderType  tenderId  submittingOffersDate  procedureResult  contractors  isTenderAmountBelowEU
noticeType                                                                                                                                
AgreementIntentionNotice              100         100       100                     0                0          100                    100
AgreementUpdateNotice                 100         100       100                     0                0          100                    100
CircumstancesFulfillmentNotice        100         100       100                     0                0          100                    100
CompetitionNotice                       0         100       100                     0                0            0                    100
ConcessionNotice                        0         100       100                   100                0            0                

## 4. Code fields: clientType, orderType, tenderType

These are coded values — what do they look like, how many distinct values,
and how do they relate to notice types?

In [16]:
for col in ["clientType", "orderType", "tenderType"]:
    vc = df[col].value_counts(dropna=False)
    print(f"\n--- {col} ({vc.shape[0]} unique) ---")
    print(vc.head(20))


--- clientType (41 unique) ---
clientType
1.1.2      40744
1.1.12     12967
1.4         9683
1.1.5       8778
1.5         5587
1.1.13      5358
1.1.1.1     4623
1.2         2818
1.1.15      1298
1.1.16      1056
1.1.1.2      849
1.1.6        763
1.1.7        587
1.1.3        521
1.1.14       309
1.1.10       265
1.3          184
1.1.8        163
2.1.1        152
2.1.4        127
Name: count, dtype: int64

--- orderType (4 unique) ---
orderType
Delivery    40838
Works       23207
Services    21127
None        12177
Name: count, dtype: int64

--- tenderType (46 unique) ---
tenderType
1.1.1       68199
1.1.2       13880
2.1         10764
1.4.2         991
1.4.1.7       825
1.4.1.1       684
None          549
1.4.1.11      316
2.8.1         291
1.4.1.5       208
11.1.1        104
2.7.11         80
2.7.1          64
1.4.1.2        58
1.4.3          58
2.7.7          49
2.7.6          41
2.7.5          24
1.4.1.12       16
11.1.2         15
Name: count, dtype: int64


In [17]:
# Cross-tab: orderType vs noticeType
print("orderType × noticeType (counts):")
ct = pd.crosstab(df["orderType"].fillna("(null)"), df["noticeType"])
# Show only types with >50 records
ct = ct.loc[:, ct.sum() > 50]
print(ct)

orderType × noticeType (counts):
noticeType  AgreementIntentionNotice  AgreementUpdateNotice  ContractNotice  \
orderType                                                                     
(null)                             0                      0               0   
Delivery                         166                    159           12707   
Services                         419                    212            7238   
Works                            265                    876            5972   

noticeType  ContractPerformingNotice  NoticeUpdateNotice  SmallContractNotice  \
orderType                                                                       
(null)                             0               12132                   40   
Delivery                       18773                   0                  157   
Services                        6874                   0                  229   
Works                           7293                   0                  123   

notic

In [18]:
# Which notice types have null orderType?
null_order = df[df["orderType"].isna()]
print(f"Records with null orderType: {len(null_order):,}")
print(null_order["noticeType"].value_counts())

Records with null orderType: 12,177
noticeType
NoticeUpdateNotice     12132
SmallContractNotice       40
CompetitionNotice          4
ConcessionNotice           1
Name: count, dtype: int64


## 5. Field completeness (overall + by notice type)

In [19]:
null_pct = df.drop(columns=["_file_date", "_pub_dt", "_weekday", "_pub_hour"]).isnull().mean() * 100
null_pct = null_pct.sort_values(ascending=False)
print("Overall null % per column:")
for col, pct in null_pct.items():
    marker = " ← conditional" if pct > 5 else ""
    print(f"  {col:30s} {pct:5.1f}%{marker}")

Overall null % per column:
  procedureResult                 75.7% ← conditional
  submittingOffersDate            73.1% ← conditional
  contractors                     39.7% ← conditional
  orderType                       12.5% ← conditional
  tenderType                       0.6%
  tenderId                         0.6%
  clientType                       0.1%
  organizationProvince             0.0%
  orderObject                      0.0%
  publicationDate                  0.0%
  cpvCode                          0.0%
  isTenderAmountBelowEU            0.0%
  bzpNumber                        0.0%
  organizationName                 0.0%
  organizationCity                 0.0%
  organizationCountry              0.0%
  organizationNationalId           0.0%
  organizationId                   0.0%
  noticeNumber                     0.0%
  htmlBody                         0.0%
  noticeType                       0.0%
  objectId                         0.0%


## 6. HTML structure inventory

Different notice types have different HTML templates. Let's see what
`<h2>` section headers exist in each type — this tells us what we
*could* parse.

In [20]:
def extract_h2_sections(html: str) -> list[str]:
    """Return all <h2> section headers from a BZP notice HTML."""
    soup = BeautifulSoup(html, "lxml")
    return [h2.get_text(strip=True) for h2 in soup.find_all("h2")]

# Sample a few records from each notice type
for ntype in df["noticeType"].value_counts().head(6).index:
    subset = df[df["noticeType"] == ntype]
    sample_html = subset["htmlBody"].iloc[0]
    sections = extract_h2_sections(sample_html)
    print(f"\n=== {ntype} ({len(subset):,} records) ===")
    for s in sections:
        print(f"  {s}")


=== ContractPerformingNotice (32,940 records) ===
  SEKCJA I - ZAMAWIAJĄCY
  SEKCJA II – INFORMACJE PODSTAWOWE
  SEKCJA III – PODSTAWOWE INFORMACJE O POSTĘPOWANIU W WYNIKU KTÓREGO ZOSTAŁA ZAWARTA UMOWA
  SEKCJA IV – PODSTAWOWE INFORMACJE O ZAWARTEJ UMOWIE
  SEKCJA V PRZEBIEG REALIZACJI UMOWY
  SEKCJA VI INFORMACJE DODATKOWE

=== ContractNotice (25,917 records) ===
  SEKCJA I - ZAMAWIAJĄCY
  SEKCJA II – INFORMACJE PODSTAWOWE
  SEKCJA III – UDOSTĘPNIANIE DOKUMENTÓW ZAMÓWIENIA I KOMUNIKACJA
  SEKCJA IV – PRZEDMIOT ZAMÓWIENIA
  SEKCJA V - KWALIFIKACJA WYKONAWCÓW
  SEKCJA VI - WARUNKI ZAMÓWIENIA
  SEKCJA VII - PROJEKTOWANE POSTANOWIENIA UMOWY
  SEKCJA VIII – PROCEDURA

=== TenderResultNotice (23,675 records) ===
  SEKCJA I - ZAMAWIAJĄCY
  SEKCJA II – INFORMACJE PODSTAWOWE
  SEKCJA III – TRYB UDZIELENIA ZAMÓWIENIA LUB ZAWARCIA UMOWY RAMOWEJ
  SEKCJA IV – PRZEDMIOT ZAMÓWIENIA
  SEKCJA V ZAKOŃCZENIE POSTĘPOWANIA
  SEKCJA VI OFERTY
  SEKCJA VII WYKONAWCA, KTÓREMU UDZIELONO ZAMÓWIENIA
  SEKCJA 

In [21]:
def extract_h3_fields(html: str) -> list[str]:
    """Return all numbered field labels (e.g. '1.5.1.') from <h3> tags."""
    soup = BeautifulSoup(html, "lxml")
    fields = []
    for h3 in soup.find_all("h3"):
        text = h3.get_text(strip=True)
        # Extract the field number pattern like "1.5.1.)"
        m = re.match(r"([\d.]+\.)\)", text)
        if m:
            fields.append(m.group(1))
    return fields

# Compare field numbers across the top 4 notice types
top_types = df["noticeType"].value_counts().head(4).index
type_fields = {}
for ntype in top_types:
    # Sample a few and collect the union of fields
    samples = df[df["noticeType"] == ntype]["htmlBody"].head(5)
    all_fields = set()
    for html in samples:
        all_fields.update(extract_h3_fields(html))
    type_fields[ntype] = sorted(all_fields)
    print(f"\n{ntype}: {len(all_fields)} fields")
    print(f"  {', '.join(sorted(all_fields)[:30])}{'...' if len(all_fields)>30 else ''}")


ContractPerformingNotice: 48 fields
  1.1., 1.3., 1.4.1., 1.4.10., 1.4.2., 1.4.3., 1.4.4., 1.4.5., 1.4.6., 1.4.7., 1.4.8., 1.4.9., 1.5., 2.1., 2.2., 2.3., 2.4., 3.1., 3.10., 3.2., 3.2.1., 3.3., 3.5., 3.6., 3.7., 3.8., 3.9., 4.1., 4.2., 4.3....

ContractNotice: 89 fields
  1.1., 1.2., 1.5.1., 1.5.10., 1.5.2., 1.5.3., 1.5.4., 1.5.5., 1.5.6., 1.5.7., 1.5.8., 1.5.9., 1.6., 1.7., 2.1., 2.10., 2.11., 2.14., 2.16., 2.2., 2.3., 2.4., 2.5., 2.6., 2.7., 2.8., 2.9., 3.1., 3.12., 3.14....

TenderResultNotice: 63 fields
  1.1., 1.2., 1.3., 1.5.1., 1.5.10., 1.5.2., 1.5.3., 1.5.4., 1.5.5., 1.5.6., 1.5.7., 1.5.9., 1.6., 1.7., 1.8., 2.1., 2.10., 2.11., 2.13., 2.14., 2.2., 2.3., 2.4., 2.5., 2.6., 2.7., 2.8., 2.9., 3.1., 4.1....

NoticeUpdateNotice: 21 fields
  1.1., 1.3., 1.4., 1.4.1., 1.4.10., 1.4.2., 1.4.3., 1.4.4., 1.4.5., 1.4.6., 1.4.7., 1.4.8., 1.4.9., 1.5., 1.6., 2.1., 2.2., 3.2., 3.3., 3.4., 3.4.1.


## 7. HTML extraction quality by notice type

Run the current parser on a stratified sample and measure per-type
extraction rates.

In [22]:
# Stratified sample: up to 200 per notice type
strat = df.groupby("noticeType").apply(
    lambda g: g.sample(min(200, len(g)), random_state=42),
    include_groups=False,
).reset_index(level=0)
print(f"Stratified sample: {len(strat)} records")

strat["_parsed"] = strat["htmlBody"].apply(
    lambda h: parse_html(h).model_dump()
    if isinstance(h, str) and h.rstrip().endswith("</html>") else None
)
df_p = pd.json_normalize(strat["_parsed"].dropna())
df_p["noticeType"] = strat.loc[strat["_parsed"].notna(), "noticeType"].values

extract_cols = ["ulica", "kod_pocztowy", "nuts3_code", "opis", "kryteria_oceny", "wartosc_umowy_pln"]
rates = df_p.groupby("noticeType")[extract_cols].apply(lambda g: g.notna().mean() * 100)
print("\nExtraction rate (%) per notice type:")
print(rates.round(0).astype(int).to_string())

Stratified sample: 1439 records

Extraction rate (%) per notice type:
                                ulica  kod_pocztowy  nuts3_code  opis  kryteria_oceny  wartosc_umowy_pln
noticeType                                                                                              
AgreementIntentionNotice          100           100         100     0               0                  0
AgreementUpdateNotice               0             0           0     0               0                  0
CircumstancesFulfillmentNotice      0             0           0     0               0                  0
CompetitionNotice                 100           100         100     0               0                  0
ConcessionNotice                    0             0           0     0               0                  0
ContractNotice                    100           100         100   100              89                  2
ContractPerformingNotice            0             0           0     0               0     

## 8. CPV code analysis

In [23]:
df["cpvCodes"] = df["cpvCode"].apply(
    lambda x: parse_cpv_codes(x) if pd.notna(x) else []
)
df["cpv_count"] = df["cpvCodes"].apply(len)

print("CPV codes per notice:")
print(df["cpv_count"].describe().to_string())

# Group by 2-digit CPV division for readability
all_cpv = df["cpvCodes"].explode().dropna()
cpv_div = all_cpv.str[:2]

CPV_DIVISIONS = {
    "03": "Agriculture", "09": "Petroleum/fuel", "14": "Mining",
    "15": "Food", "18": "Clothing", "22": "Printed matter",
    "24": "Chemicals", "30": "Office/computing", "31": "Electrical",
    "32": "Radio/TV/telecom", "33": "Medical", "34": "Transport equip",
    "35": "Security/defence", "37": "Musical/sports", "38": "Lab equip",
    "39": "Furniture", "42": "Industrial machinery", "44": "Construction materials",
    "45": "Construction works", "48": "Software", "50": "Repair/maintenance",
    "55": "Hotel/restaurant", "60": "Transport", "63": "Travel agency",
    "64": "Postal/telecom", "65": "Utilities", "66": "Financial",
    "70": "Real estate", "71": "Architecture/engineering", "72": "IT services",
    "73": "R&D", "75": "Public admin", "76": "Oil/gas",
    "77": "Agriculture services", "79": "Business services", "80": "Education",
    "85": "Health/social", "90": "Environment", "92": "Recreation/culture",
    "98": "Other services",
}

div_counts = cpv_div.value_counts().head(15)
print("\nTop 15 CPV divisions:")
for code, count in div_counts.items():
    label = CPV_DIVISIONS.get(code, "?")
    print(f"  {code} {label:30s} {count:>6,}")

CPV codes per notice:
count    97349.000000
mean         2.742771
std          3.961656
min          0.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        120.000000

Top 15 CPV divisions:
  45 Construction works             114,689
  33 Medical                        20,009
  71 Architecture/engineering       14,215
  39 Furniture                      10,882
  30 Office/computing                9,153
  15 Food                            9,026
  90 Environment                     8,646
  34 Transport equip                 7,629
  31 Electrical                      5,970
  48 Software                        5,242
  32 Radio/TV/telecom                4,121
  55 Hotel/restaurant                3,992
  44 Construction materials          3,942
  80 Education                       3,776
  79 Business services               3,577


## 9. procedureResult values

In [24]:
print("procedureResult distribution (non-null only):")
pr = df["procedureResult"].dropna()
print(f"  non-null: {len(pr):,} / {len(df):,} ({len(pr)/len(df)*100:.1f}%)")
print(pr.value_counts())

print("\nprocedureResult by notice type:")
print(pd.crosstab(df["procedureResult"].fillna("(null)"), df["noticeType"]))

procedureResult distribution (non-null only):
  non-null: 23,675 / 97,349 (24.3%)
procedureResult
zawarcieUmowy                                                                                                                                                                                                                                                                                            14338
uniewaznienie                                                                                                                                                                                                                                                                                             3516
zawarcieUmowy;zawarcieUmowy                                                                                                                                                                                                                                                                             

## 10. Contract values (from TenderResultNotice HTML)

In [25]:
# Parse contract values from TenderResultNotice only
tender_results = df[df["noticeType"] == "TenderResultNotice"].copy()
tender_results["_value"] = tender_results["htmlBody"].apply(
    lambda h: parse_html(h).wartosc_umowy_pln
    if isinstance(h, str) and h.rstrip().endswith("</html>") else None
)

values = tender_results["_value"].dropna()
print(f"Contract values extracted: {len(values):,} / {len(tender_results):,} TenderResultNotice")
print("\nDistribution (PLN):")
print(values.describe().apply(lambda x: f"{x:,.0f}"))

print("\nPercentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    v = values.quantile(p/100)
    print(f"  P{p:2d}: {v:>15,.0f} PLN")

Contract values extracted: 19,317 / 23,675 TenderResultNotice

Distribution (PLN):
count           19,317
mean         1,171,667
std         54,384,949
min                  0
25%             98,790
50%            231,855
75%            479,700
max      5,342,169,325
Name: _value, dtype: object

Percentiles:
  P 1:           2,848 PLN
  P 5:          13,278 PLN
  P10:          29,520 PLN
  P25:          98,790 PLN
  P50:         231,855 PLN
  P75:         479,700 PLN
  P90:       1,038,774 PLN
  P95:       2,148,895 PLN
  P99:       7,825,359 PLN


## 11. Organizations & geography

In [26]:
PROVINCE_NAMES = {
    "PL02": "Dolnośląskie", "PL04": "Kujawsko-Pomorskie",
    "PL06": "Lubelskie", "PL08": "Lubuskie",
    "PL10": "Łódzkie", "PL12": "Małopolskie",
    "PL14": "Mazowieckie", "PL16": "Opolskie",
    "PL18": "Podkarpackie", "PL20": "Podlaskie",
    "PL22": "Pomorskie", "PL24": "Śląskie",
    "PL26": "Świętokrzyskie", "PL28": "Warmińsko-Mazurskie",
    "PL30": "Wielkopolskie", "PL32": "Zachodniopomorskie",
}

prov = df["organizationProvince"].value_counts()
print("Notices by province:")
for code, count in prov.items():
    name = PROVINCE_NAMES.get(code, code)
    print(f"  {code} {name:25s} {count:>6,}  ({count/len(df)*100:.1f}%)")

print(f"\nUnique organizations: {df['organizationName'].nunique():,}")
print(f"Unique cities: {df['organizationCity'].nunique():,}")
print(f"Unique NIP (tax IDs): {df['organizationNationalId'].nunique():,}")

Notices by province:
  PL14 Mazowieckie               17,145  (17.6%)
  PL24 Śląskie                    9,502  (9.8%)
  PL12 Małopolskie                8,855  (9.1%)
  PL30 Wielkopolskie              7,685  (7.9%)
  PL02 Dolnośląskie               6,989  (7.2%)
  PL06 Lubelskie                  5,930  (6.1%)
  PL18 Podkarpackie               5,900  (6.1%)
  PL22 Pomorskie                  5,673  (5.8%)
  PL10 Łódzkie                    5,445  (5.6%)
  PL04 Kujawsko-Pomorskie         4,975  (5.1%)
  PL32 Zachodniopomorskie         3,976  (4.1%)
  PL28 Warmińsko-Mazurskie        3,703  (3.8%)
  PL20 Podlaskie                  3,545  (3.6%)
  PL26 Świętokrzyskie             3,371  (3.5%)
  PL08 Lubuskie                   2,322  (2.4%)
  PL16 Opolskie                   2,293  (2.4%)

Unique organizations: 7,880
Unique cities: 2,453
Unique NIP (tax IDs): 7,870


In [27]:
# Top publishers by notice type — who publishes what?
top_orgs = df["organizationName"].value_counts().head(10).index
ct = pd.crosstab(df.loc[df["organizationName"].isin(top_orgs), "organizationName"],
                 df["noticeType"])
# Keep only main types
ct = ct.loc[:, ct.sum() > 0]
print("Top 10 organizations × notice type:")
print(ct)

Top 10 organizations × notice type:
noticeType                                                                       AgreementIntentionNotice  \
organizationName                                                                                            
Agencja Mienia Wojskowego                                                                               0   
Akademia Górniczo-Hutnicza im. Stanisława Staszica w Krakowie                                           0   
Mazowiecki Szpital Wojewódzki mi. św. Jana Pawła II w Siedlcach Sp. z o.o.                              0   
POLITECHNIKA WARSZAWSKA                                                                                 2   
Państwowe Gospodarstwo Wodne Wody Polskie                                                               0   
SAMODZIELNY PUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ SZPITAL UNIWERSYTECKI W KRAKOWIE                         0   
Sosnowiecki Szpital Miejski Sp. z o.o. w restrukturyzacji                                   

## 12. Contractors deep dive

In [28]:
# Only records that have contractors
has_c = df[df["contractors"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()
has_c["_n_contractors"] = has_c["contractors"].apply(len)

print(f"Records with contractors: {len(has_c):,} / {len(df):,}")
print("\nContractors per record:")
print(has_c["_n_contractors"].describe().to_string())

print("\nDistribution:")
print(has_c["_n_contractors"].value_counts().sort_index().head(10))

Records with contractors: 58,746 / 97,349

Contractors per record:
count    58746.000000
mean         1.401219
std          2.341837
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         98.000000

Distribution:
_n_contractors
1     52925
2      2264
3      1052
4       691
5       421
6       305
7       221
8       162
9       128
10      116
Name: count, dtype: int64


In [29]:
# Contractor field completeness
c_flat = has_c["contractors"].explode().apply(pd.Series)
print("Contractor field completeness:")
for col in c_flat.columns:
    rate = c_flat[col].notna().mean() * 100
    print(f"  {col:30s} {rate:5.1f}%")

Contractor field completeness:
  contractorName                  58.4%
  contractorCity                  58.4%
  contractorProvince              50.8%
  contractorCountry               58.4%
  contractorNationalId            57.0%


## 13. isTenderAmountBelowEU flag

In [30]:
print("isTenderAmountBelowEU distribution:")
print(df["isTenderAmountBelowEU"].value_counts())

print("\nBy notice type:")
ct = pd.crosstab(df["noticeType"], df["isTenderAmountBelowEU"], normalize="index") * 100
print(ct.round(1))

isTenderAmountBelowEU distribution:
isTenderAmountBelowEU
True     85958
False    11391
Name: count, dtype: int64

By notice type:
isTenderAmountBelowEU           False  True 
noticeType                                  
AgreementIntentionNotice          4.4   95.6
AgreementUpdateNotice             0.0  100.0
CircumstancesFulfillmentNotice   52.9   47.1
CompetitionNotice                 0.0  100.0
ConcessionNotice                  0.0  100.0
ContractNotice                    0.0  100.0
ContractPerformingNotice         33.4   66.6
NoticeUpdateNotice                2.5   97.5
SmallContractNotice               0.0  100.0
TenderResultNotice                0.1   99.9


## 14. Linking notices: bzpNumber as a join key

Multiple notice types can share a `bzpNumber` (e.g. a ContractNotice
followed by a TenderResultNotice for the same tender). How common is this?

In [31]:
bzp_types = df.groupby("bzpNumber")["noticeType"].apply(set)
bzp_multi = bzp_types[bzp_types.apply(len) > 1]

print(f"Unique bzpNumbers: {df['bzpNumber'].nunique():,}")
print(f"bzpNumbers with multiple notice types: {len(bzp_multi):,}")

# Most common type combinations
combos = bzp_multi.apply(lambda s: " + ".join(sorted(s)))
print("\nTop type combinations sharing a bzpNumber:")
print(combos.value_counts().head(10))

Unique bzpNumbers: 51,379
bzpNumbers with multiple notice types: 0

Top type combinations sharing a bzpNumber:
Series([], Name: count, dtype: int64)


In [32]:
# How many notices per bzpNumber?
notices_per_bzp = df.groupby("bzpNumber").size()
print("Notices per bzpNumber:")
print(notices_per_bzp.describe().to_string())
print("\nDistribution:")
print(notices_per_bzp.value_counts().sort_index().head(10))

Notices per bzpNumber:
count    51379.000000
mean         1.894724
std          0.306912
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max          2.000000

Distribution:
1     5409
2    45970
Name: count, dtype: int64
